## 0. One-time setup

In [78]:
# Install dependencies (run once). Restart the kernel after installing if prompted.
%pip install -q google-adk google-cloud-aiplatform[adk] google-genai litellm requests

## 1. Setup, Installation and API Key management

In [79]:
import os
from getpass import getpass

from google.genai import types
from google.adk.models import Gemini
from google.adk.runners import InMemoryRunner
from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm # For multi-model support

# Vertex AI auth for Gemini (project-based, not an API key).
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
PROJECT_ID = "qwiklabs-gcp-03-8f57c8b00ccc"
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ.setdefault("GOOGLE_CLOUD_LOCATION", "us-central1")

MODEL_NAME = os.getenv("MODEL", "gemini-2.5-flash")

# Interactive password-style prompts keep keys out of any file on disk.
if not os.environ.get("GOOGLE_MAPS_API_KEY"):
    os.environ["GOOGLE_MAPS_API_KEY"] = getpass("Enter your Google Maps API key: ")
GOOGLE_MAPS_API_KEY = os.getenv("GOOGLE_MAPS_API_KEY")

if not os.environ.get("OPENAI_API_KEY"):
    entered_openai_key = getpass(
        "Enter your OpenAI API key (leave blank to skip the GPT model variant): "
    )
    if entered_openai_key:
        os.environ["OPENAI_API_KEY"] = entered_openai_key
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")

if not GOOGLE_MAPS_API_KEY:
    print("WARNING: GOOGLE_MAPS_API_KEY is not set. The geocoding tool will not work without it.")
if not OPENAI_API_KEY:
    print("NOTE: OPENAI_API_KEY is not set. The GPT model variant will be skipped in later cells.")

# Retry options help avoid the occasional error from popular models
# receiving too many requests at once.
RETRY_OPTIONS = types.HttpRetryOptions(initial_delay=1, max_delay=3, attempts=30)\

print(
    f"Setup complete. PROJECT_ID={PROJECT_ID!r}, MODEL_NAME={MODEL_NAME!r}, "
    f"GOOGLE_MAPS_API_KEY set={bool(GOOGLE_MAPS_API_KEY)}, OPENAI_API_KEY set={bool(OPENAI_API_KEY)}"
)

Enter your OpenAI API key (leave blank to skip the GPT model variant): ··········
NOTE: OPENAI_API_KEY is not set. The GPT model variant will be skipped in later cells.
Setup complete. PROJECT_ID='qwiklabs-gcp-03-8f57c8b00ccc', MODEL_NAME='gemini-2.5-flash', GOOGLE_MAPS_API_KEY set=True, OPENAI_API_KEY set=False


# 2. Tool: Google Maps Geocoding

Converts a place name (e.g. `"Seattle, WA"`) into latitude/longitude using the
[Google Maps Geocoding API](https://developers.google.com/maps/documentation/geocoding).

In [81]:
import requests


def get_lat_long_for_place(place: str) -> dict[str, float | str]:
    """Convert a place name to latitude/longitude using the Google Maps Geocoding API.

    Args:
        place: A place description, e.g. "Seattle, WA" or "1600 Amphitheatre Parkway,
            Mountain View, CA".

    Returns:
        On success: {"status": "success", "latitude": float, "longitude": float,
        "formatted_address": str}.
        On failure: {"status": "error", "error_message": str}.
    """
    print(f"[get_lat_long_for_place] Geocoding {place!r}...")

    if not GOOGLE_MAPS_API_KEY:
        return {"status": "error", "error_message": "GOOGLE_MAPS_API_KEY is not set."}

    try:
        response = requests.get(
            "https://maps.googleapis.com/maps/api/geocode/json",
            params={"address": place, "key": GOOGLE_MAPS_API_KEY},
            timeout=10,
        )
        response.raise_for_status()
        data = response.json()

        if data.get("status") != "OK" or not data.get("results"):
            print(f"[get_lat_long_for_place] Geocoding API returned: {data.get('status')}")
            return {
                "status": "error",
                "error_message": f"Geocoding API returned: {data.get('status')}",
            }

        result = data["results"][0]
        location = result["geometry"]["location"]
        print(
            f"[get_lat_long_for_place] Resolved to ({location['lat']}, {location['lng']}) "
            f"-- {result['formatted_address']}"
        )
        return {
            "status": "success",
            "latitude": location["lat"],
            "longitude": location["lng"],
            "formatted_address": result["formatted_address"],
        }
    except requests.RequestException as exc:
        print(f"[get_lat_long_for_place] Request failed: {exc}")
        return {"status": "error", "error_message": f"Geocoding request failed: {exc}"}

# Test Seattle, WA
print(get_lat_long_for_place("Seattle, WA"))

[get_lat_long_for_place] Geocoding 'Seattle, WA'...
[get_lat_long_for_place] Resolved to (47.6061389, -122.3328481) -- Seattle, WA, USA
{'status': 'success', 'latitude': 47.6061389, 'longitude': -122.3328481, 'formatted_address': 'Seattle, WA, USA'}


## 3. Tool: National Weather Service current conditions and alerts

Uses the free, keyless [National Weather Service API](https://www.weather.gov/documentation/services-web-api)
to look up the forecast zone for a coordinate, fetch the nearest forecast period, and check for
any active weather alerts covering that point.

NWS requires a descriptive `User-Agent` header identifying the application on every request.

In [82]:
NWS_HEADERS = {"User-Agent": "weather-agent-demo (contact: samuel.k.imlig@saic.com)"}


def get_weather_by_coordinates(latitude: float, longitude: float) -> dict:
    """Get current forecast conditions and active alerts for a coordinate via the NWS API.

    Args:
        latitude: Latitude in decimal degrees, e.g. 47.6062.
        longitude: Longitude in decimal degrees, e.g. -122.3321.

    Returns:
        On success: {"status": "success", "forecast_summary": str,
        "active_alerts": list[str]} where active_alerts is empty if there are none.
        On failure: {"status": "error", "error_message": str}.
    """
    print(f"[get_weather_by_coordinates] Looking up forecast zone for ({latitude}, {longitude})...")

    try:
        points_resp = requests.get(
            f"https://api.weather.gov/points/{latitude},{longitude}",
            headers=NWS_HEADERS,
            timeout=10,
        )
        points_resp.raise_for_status()
        forecast_url = points_resp.json()["properties"]["forecast"]
        print(f"[get_weather_by_coordinates] Forecast URL: {forecast_url}")

        forecast_resp = requests.get(forecast_url, headers=NWS_HEADERS, timeout=10)
        forecast_resp.raise_for_status()
        periods = forecast_resp.json()["properties"]["periods"]
        current_period = periods[0]
        forecast_summary = (
            f"{current_period['name']}: {current_period['detailedForecast']}"
        )
        print(f"[get_weather_by_coordinates] Forecast: {forecast_summary}")

        alerts_resp = requests.get(
            "https://api.weather.gov/alerts/active",
            params={"point": f"{latitude},{longitude}"},
            headers=NWS_HEADERS,
            timeout=10,
        )
        alerts_resp.raise_for_status()
        alert_features = alerts_resp.json().get("features", [])
        active_alerts = [
            feature["properties"]["headline"]
            for feature in alert_features
            if feature.get("properties", {}).get("headline")
        ]
        print(f"[get_weather_by_coordinates] Active alerts: {active_alerts or 'none'}")

        return {
            "status": "success",
            "forecast_summary": forecast_summary,
            "active_alerts": active_alerts,
        }
    except requests.RequestException as exc:
        print(f"[get_weather_by_coordinates] Request failed: {exc}")
        return {"status": "error", "error_message": f"NWS request failed: {exc}"}
    except (KeyError, IndexError) as exc:
        print(f"[get_weather_by_coordinates] Unexpected response shape: {exc}")
        return {"status": "error", "error_message": f"Unexpected NWS response shape: {exc}"}

# Quick manual check: chain Tool 2 -> Tool 3 for a single city before wiring up the agent.
seattle_location = get_lat_long_for_place("Seattle, WA")
print("Tool 2 result:", seattle_location)

if seattle_location["status"] == "success":
    seattle_weather = get_weather_by_coordinates(
        seattle_location["latitude"], seattle_location["longitude"]
    )
    print("Tool 3 result:", seattle_weather)
else:
    print("Skipping Tool 3 call -- geocoding failed.")

[get_lat_long_for_place] Geocoding 'Seattle, WA'...
[get_lat_long_for_place] Resolved to (47.6061389, -122.3328481) -- Seattle, WA, USA
Tool 2 result: {'status': 'success', 'latitude': 47.6061389, 'longitude': -122.3328481, 'formatted_address': 'Seattle, WA, USA'}
[get_weather_by_coordinates] Looking up forecast zone for (47.6061389, -122.3328481)...
[get_weather_by_coordinates] Forecast URL: https://api.weather.gov/gridpoints/SEW/125,68/forecast
[get_weather_by_coordinates] Forecast: Today: Sunny, with a high near 83. Northwest wind 1 to 5 mph.
[get_weather_by_coordinates] Active alerts: ['Heat Advisory issued August 6 at 8:11AM PDT until August 7 at 10:00PM PDT by NWS Seattle WA', 'Air Quality Alert issued August 5 at 3:47PM PDT by NWS Seattle WA']
Tool 3 result: {'status': 'success', 'forecast_summary': 'Today: Sunny, with a high near 83. Northwest wind 1 to 5 mph.', 'active_alerts': ['Heat Advisory issued August 6 at 8:11AM PDT until August 7 at 10:00PM PDT by NWS Seattle WA', 'Air

## 4. Weather agent

`build_weather_agent` creates an `LlmAgent` wired to both tools, given a model. The same
instruction and tool set is reused for both the Gemini and GPT variants below, so the two
agents are directly comparable in the multi-city tests.

In [83]:
import vertexai
from vertexai.preview import reasoning_engines

vertexai.init(project=PROJECT_ID, location=os.environ["GOOGLE_CLOUD_LOCATION"])

WEATHER_AGENT_INSTRUCTION = """
You are a weather assistant. For every user request about weather in a place:

1. Call get_lat_long_for_place to convert the place name into latitude/longitude.
   If that fails, tell the user you could not find the location and stop.
2. Call get_weather_by_coordinates with those coordinates.
   If that fails, tell the user the weather lookup failed and stop.
3. If active_alerts is non-empty, lead your reply with "ALERT:" followed by the
   alert headline(s), then give a brief weather summary.
4. If active_alerts is empty, give a short, friendly weather summary based on
   forecast_summary -- no need to mention alerts explicitly.

Always name the location in your reply.
"""


def build_weather_agent(name: str, model) -> Agent:
    """Create a weather LlmAgent wired to the geocoding and NWS tools.

    Args:
        name: Unique agent name.
        model: A model identifier string, or a Gemini/LiteLlm model object.

    Returns:
        A configured LlmAgent ready to run.
    """
    return Agent(
        name=name,
        description="Provides current weather summaries and alerts for US locations.",
        model=model,
        instruction=WEATHER_AGENT_INSTRUCTION,
        tools=[get_lat_long_for_place, get_weather_by_coordinates],
    )


gemini_weather_agent = reasoning_engines.AdkApp(
    agent=build_weather_agent("gemini_weather_agent", Gemini(model=MODEL_NAME, retry_options=RETRY_OPTIONS)),
    enable_tracing=False,
)

gpt_weather_agent = (
    reasoning_engines.AdkApp(
        agent=build_weather_agent("gpt_weather_agent", LiteLlm(model="openai/gpt-4o-mini")),
        enable_tracing=False,
    )
    if OPENAI_API_KEY
    else None
)

print("gemini_weather_agent ready:", gemini_weather_agent)
print("gpt_weather_agent ready:", gpt_weather_agent if gpt_weather_agent else "SKIPPED (no OPENAI_API_KEY)")

gemini_weather_agent ready: <vertexai.preview.reasoning_engines.templates.adk.AdkApp object at 0x7c0777060b00>
gpt_weather_agent ready: SKIPPED (no OPENAI_API_KEY)


## 5. Create Test Session

In [84]:
app_user_id = "test-user"
app_session = gemini_weather_agent.create_session(user_id=app_user_id)

print("AdkApp session id:", app_session["id"])
print("AdkApp session:", app_session)

This legacy setting overrides the new Cloud Console toggle and environment variable controls.
Impact: The Cloud Console may incorrectly show telemetry as 'On' when it is actually 'Off', and the UI toggle will not work.
Action: To fix this and control telemetry, please remove the 'enable_tracing' parameter from your deployment code.
You can then use the 'GOOGLE_CLOUD_AGENT_ENGINE_ENABLE_TELEMETRY' environment variable:
agent_engines.create(
  env_vars={
    "GOOGLE_CLOUD_AGENT_ENGINE_ENABLE_TELEMETRY": true|false
  }
)
or the toggle in the Cloud Console: https://console.cloud.google.com/vertex-ai/agents.


AdkApp session id: 0aa17d7d-14f1-4797-ba5c-3c0ad77a9865
AdkApp session: {'id': '0aa17d7d-14f1-4797-ba5c-3c0ad77a9865', 'app_name': 'default-app-name', 'user_id': 'test-user', 'state': {}, 'events': [], 'last_update_time': 1786031790.787968}


## 6. Ask Function

In [88]:
def ask(adk_app: "reasoning_engines.AdkApp", query: str, user_id: str) -> str:
    """Send one user message through an AdkApp-wrapped agent and return the final response text."""
    print(f"[user] Sending to session for {user_id!r}: {query!r}")
    session = adk_app.create_session(user_id=user_id)

    final_text = ""
    for event in adk_app.stream_query(
        user_id=user_id, session_id=session["id"], message=query
    ):
        for part in event.get("content", {}).get("parts", []):
            if part.get("text"):
                final_text += part["text"]
    print(f"[ask] Done for session {session['id']!r}")
    return final_text

## 7. Unit tests for the tool functions (no LLM calls)

Pure tests against the geocoding and NWS functions directly, independent of any agent or model.

In [89]:
def test_get_lat_long_for_place():
    if not GOOGLE_MAPS_API_KEY:
        print("SKIPPED test_get_lat_long_for_place: GOOGLE_MAPS_API_KEY not set")
        return
    result = get_lat_long_for_place("Seattle, WA")
    assert result["status"] == "success"
    assert "latitude" in result and "longitude" in result
    print("test_get_lat_long_for_place passed:", result)


def test_get_weather_by_coordinates():
    # Seattle, WA coordinates -- used directly so this test does not depend on the geocoding tool.
    result = get_weather_by_coordinates(47.6062, -122.3321)
    assert result["status"] == "success"
    assert "forecast_summary" in result and "active_alerts" in result
    print("test_get_weather_by_coordinates passed:", result)


test_get_lat_long_for_place()
test_get_weather_by_coordinates()

[get_lat_long_for_place] Geocoding 'Seattle, WA'...
[get_lat_long_for_place] Resolved to (47.6061389, -122.3328481) -- Seattle, WA, USA
test_get_lat_long_for_place passed: {'status': 'success', 'latitude': 47.6061389, 'longitude': -122.3328481, 'formatted_address': 'Seattle, WA, USA'}
[get_weather_by_coordinates] Looking up forecast zone for (47.6062, -122.3321)...
[get_weather_by_coordinates] Forecast URL: https://api.weather.gov/gridpoints/SEW/125,68/forecast
[get_weather_by_coordinates] Forecast: Today: Sunny, with a high near 83. Northwest wind 1 to 5 mph.
[get_weather_by_coordinates] Active alerts: ['Heat Advisory issued August 6 at 8:11AM PDT until August 7 at 10:00PM PDT by NWS Seattle WA', 'Air Quality Alert issued August 5 at 3:47PM PDT by NWS Seattle WA']
test_get_weather_by_coordinates passed: {'status': 'success', 'forecast_summary': 'Today: Sunny, with a high near 83. Northwest wind 1 to 5 mph.', 'active_alerts': ['Heat Advisory issued August 6 at 8:11AM PDT until August 7

## 8. Multi-city agent tests

Runs each agent against a handful of US cities spanning different regions and climates. Any
active alerts should surface clearly in the response; otherwise the response should be a plain
weather summary. The GPT loop is skipped automatically if `OPENAI_API_KEY` was not set.

In [91]:
TEST_CITIES = [
    "Seattle, WA",
    "Portland, OR",
    "Sherwood, OR",
    "Miami, FL",
    "Denver, CO",
    "Chicago, IL",
    "New York, NY",
]

for city in TEST_CITIES:
    response = ask(gemini_weather_agent, f"What's the weather like in {city}?", user_id=f"gemini-{city}")
    print(f"--- Gemini | {city} ---")
    print(response)
    print()

[user] Sending to session for 'gemini-Seattle, WA': "What's the weather like in Seattle, WA?"
[get_lat_long_for_place] Geocoding 'Seattle, WA'...
[get_lat_long_for_place] Resolved to (47.6061389, -122.3328481) -- Seattle, WA, USA
[get_weather_by_coordinates] Looking up forecast zone for (47.6061389, -122.3328481)...
[get_weather_by_coordinates] Forecast URL: https://api.weather.gov/gridpoints/SEW/125,68/forecast
[get_weather_by_coordinates] Forecast: Today: Sunny, with a high near 83. Northwest wind 1 to 5 mph.
[get_weather_by_coordinates] Active alerts: ['Heat Advisory issued August 6 at 8:11AM PDT until August 7 at 10:00PM PDT by NWS Seattle WA', 'Air Quality Alert issued August 5 at 3:47PM PDT by NWS Seattle WA']
[ask] Done for session '77d5ad07-e709-44ac-99e2-a7064d46b70f'
--- Gemini | Seattle, WA ---
ALERT: Heat Advisory issued August 6 at 8:11AM PDT until August 7 at 10:00PM PDT by NWS Seattle WA, Air Quality Alert issued August 5 at 3:47PM PDT by NWS Seattle WA. In Seattle, WA, 

In [92]:
if gpt_weather_agent is None:
    print("SKIPPED GPT multi-city test: OPENAI_API_KEY not set")
else:
    for city in TEST_CITIES:
        response = ask(gpt_weather_agent, f"What's the weather like in {city}?", user_id=f"gpt-{city}")
        print(f"--- GPT | {city} ---")
        print(response)
        print()

SKIPPED GPT multi-city test: OPENAI_API_KEY not set
